# [실습] LangChain의 다양한 채팅 템플릿



실제 어플리케이션 환경에서는 프롬프트를 매번 다시 쓰는 대신,  
프롬프트의 템플릿을 구성하고, 입력 변수의 공간을 할당하여 일관성 있는 입력을 전달합니다.  


LangChain은 다양한 형태의 프롬프트 템플릿 기능을 지원합니다.

## 실습 환경 설정

기본 라이브러리를 설치합니다.

In [9]:
!pip install rich pandas langchain_community openai langchain langchain_openai langchain_google_genai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.5/120.5 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


환경 변수 파일을 불러옵니다.

In [7]:
import os
from dotenv import load_dotenv
# OPENAI_API_KEY, GOOGLE_API_KEY
load_dotenv(override=True)

True

LLM을 정의합니다.   
API 키가 필요하므로, 실습 환경에 맞게 선택해 주세요.

In [10]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# Gemini: 무료 API 사용량 존재
# 안정적 서빙을 위해 분당 10개 설정
# 즉, 초당 약 0.167개 요청 (10/60)
# `https://aistudio.google.com/`에서 모델별 사용량 확인

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.167,  # 분당 10개 요청
    check_every_n_seconds=0.1,  # 100ms마다 체크
    max_bucket_size=10,  # 최대 버스트 크기
)


# LLM 초기화
llm = ChatOpenAI(model='gpt-5-mini', temperature=0.3, max_tokens=8192)
llm_gemini = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.7, max_tokens=8192,
                                rate_limiter=rate_limiter)

response = llm_gemini.invoke("안녕?")
response

AIMessage(content='안녕하세요! 반갑습니다. 😊', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f462d-41b0-7b81-aabd-0ae4502c4b01-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 4, 'output_tokens': 38, 'total_tokens': 42, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 33}})

In [11]:
from rich import print as rprint
rprint(response)

AIMessage(
    content='안녕하세요! 반갑습니다. 😊',
    additional_kwargs={},
    response_metadata={
        'finish_reason': 'STOP',
        'model_name': 'gemini-2.5-flash',
        'safety_ratings': [],
        'model_provider': 'google_genai'
    },
    id='lc_run--019f462d-41b0-7b81-aabd-0ae4502c4b01-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 4,
        'output_tokens': 38,
        'total_tokens': 42,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 33}
    }
)

## LangChain의 Prompt

랭체인에서 LLM에 프롬프트를 입력하는 방법은 아래 3가지가 있습니다.

1. 문자열 그대로 입력하기 (이전 방법과 동일하게)
2. 메시지 클래스 리스트 입력하기
3. 프롬프트 템플릿 구성하기

### 1) 문자열 그대로 입력하기   

문자열을 `invoke()`를 통해 전달합니다.   

In [ ]:
prompt = '''
역사상 가장 매력적이었던 악역이 나오는 영화 5개를 알려주세요.
주연 배우와 명대사, 매력적인 이유도 한 줄로 설명하세요.
출력의 마지막에는 전체 내용을 표로 표현하세요.'''

result = llm.stream(prompt)
for response in result:
    print(response.content, end='')

In [ ]:
result = llm_gemini.stream(prompt)
for response in result:
    print(response.content, end='')

역사상 가장 매력적이었던 악역이 나오는 영화 5편을 선정하고, 그들의 매력적인 이유를 설명해 드립니다.

---

### 역사상 가장 매력적인 악역 영화 5선

1.  **영화 제목:** 다크 나이트 (The Dark Knight, 2008)
    *   **악역 이름:** 조커 (Joker)
    *   **주연 배우:** 히스 레저 (Heath Ledger)
    *   **명대사:** "Why so serious?" (왜 그렇게 진지해?)
    *   **매력적인 이유:** 예측 불가능한 광기와 혼돈을 통해 기존 질서를 파괴하며, 깊은 철학적 질문을 던지는 압도적인 존재감으로 관객을 사로잡습니다.

2.  **영화 제목:** 양들의 침묵 (The Silence of the Lambs, 1991)
    *   **악역 이름:** 한니발 렉터 (Hannibal Lecter)
    *   **주연 배우:** 안소니 홉킨스 (Anthony Hopkins)
    *   **명대사:** "A census taker once tried to test me. I ate his liver with some fava beans and a nice Chianti." (나는 한 번도 인구조사관에게 시험 당한 적이 없다. 그의 간을 파바 콩과 맛있는 끼안티 와인과 함께 먹었지.)
    *   **매력적인 이유:** 고도로 지적인 통찰력과 섬뜩한 교양을 겸비하며 인간 본성의 어두운 면을 탐구하는, 매혹적인 사이코패스의 전형입니다.

3.  **영화 제목:** 바스터즈: 거친 녀석들 (Inglourious Basterds, 2009)
    *   **악역 이름:** 한스 란다 (Hans Landa)
    *   **주연 배우:** 크리스토프 왈츠 (Christoph Waltz)
    *   **명대사:** "Ooh, that's a bingo! Is that how you say it? 'That's a bingo'?" (오, 빙고! 그렇게 말하는 건가? '빙고

### 2) Message 클래스 전달하기   
클래스를 직접 생성하고 전달합니다.  
`HumanMessage, SystemMessage, AIMessage` 클래스의 리스트를 전달하면 됩니다.

In [12]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

messages=[]

messages.append(SystemMessage('당신은 매우 창의력이 뛰어납니다.'))
messages.append(HumanMessage('엔비디아와 XAI가 합병한다면 어떤 이름이 좋을까요?'))

print('======= messages=====')
for message in messages :
  rprint(message)
print('=====================')
result = llm.invoke(messages)
rprint(result)

======= messages=====


SystemMessage(content='당신은 매우 창의력이 뛰어납니다.', additional_kwargs={}, response_metadata={})

HumanMessage(
    content='엔비디아와 XAI가 합병한다면 어떤 이름이 좋을까요?',
    additional_kwargs={},
    response_metadata={}
)

AIMessage(
    content='재밌는 가정이네요 — 합병한 회사 이름은 성격(하이테크·친숙·기업용·소비자용 등)에 따라 많이 달라질 수 
있습니다. 아래에 스타일별로 후보를 제안합니다. 각 이름 뒤에 간단한 설명을 붙였고, 마지막에 제가 추천하는 톱5와 짧은
태그라인도 적어놓았습니다.\n\n주의: 실제 상표·도메인 충돌 가능성이 있으니 사용 전 반드시 확인하세요.\n\n1) 짧고 
브랜드 가능한 이름\n- NVXAI — 짧고 기억하기 쉬움\n- NVX — 심플한 약칭\n- NVIDX — NVIDIA + XAI 결합형\n- XVIDIA — 
X와 NVIDIA의 자연스러운 합성\n- NVIDAI — NVIDIA + AI(Explain) 결합형\n\n2) 기술·성능 강조형\n- ClarifyCore — 성능과
해석력 강조\n- RevealCore — 투명성·핵심 연산을 모두 암시\n- ClariCompute — 명료한 컴퓨팅\n- Explainergy — 
설명(explain) + 에너지/성능\n- InterpretiGPU — 해석 가능한 GPU 플랫폼 이미지\n\n3) 설명·투명성(Explainability) 
강조형\n- Explainix — explain + nvidia 느낌\n- XplainTech — 직관적이고 설명성 강조\n- TransparentAI — 직관적이지만 
일반명사라 사용성 점검 필요\n- RevealAI — 투명성을 강조하는 간결한 이름\n- ClarionAI — 명료함(clarion) 강조\n\n4) 
기업용·신뢰감 있는 이름\n- InsightDrive — 인사이트 생성과 연산을 동시에 암시\n- LucidCompute — 명료하고 신뢰감 있는
이미지\n- EvidentAI — 설명 가능한 근거(evidence) 강조\n- NexusExplain — 연결과 설명 둘 다 암시\n- ApexExplain — 
최고 수준의 설명력 암시\n\n5) 한국어/한글 감성 후보\n- 엔비엑스 (N-V-X) — 발음하기 쉬운 합성형\n- 엑스비다 — X + 
비다(비다→비다 발음으로 NVIDIA 연상)\n- 명확AI — 설명·투명성 직관 강조\n- 투명기술 — B2B 이미지, 직설적\n- 해설엔진
— 설명 엔진이라는 직관적 이름\n\n6) 도메인/브랜딩 친화형(짧고 기억하기 쉬운 조합)\n- NVExplain — 기업 브랜드 
연장선\n- ClarifAI — clarify + AI (발음 친숙)\n- ExplainHub — 플랫폼·생태계 이미지\n- XAIWorks — 실무 중심 
이미지\n\n톱5 추천 (제 개인적 우선순위)\n1. NVXAI — 짧고 브랜드화하기 좋음. 태그라인: “Performance. 
Transparency.”\n2. XVIDIA — 기존 브랜드 감성과 자연스럽게 결합. 태그라인: “Explainable compute, reimagined.”\n3. 
ClarifyCore — 제품군(칩·SW) 모두에 확장 가능. 태그라인: “Clarity at scale.”\n4. Explainix — 스타트업·연구 느낌 모두
소화. 태그라인: “Explainable intelligence.”\n5. RevealCore — 엔터프라이즈 신뢰도를 주기 좋음. 태그라인: “Reveal the
why.”\n\n원하시면\n- 특정 스타일(예: 소비자 친화적 / 기업용 / 연구중심)에 맞춰 후보를 10개씩 더 뽑아 드리거나,\n- 
선택한 이름으로 로고·태그라인·도메인 사용 가능성(간단한 체크)까지 함께 검토해 드릴게요. 어떤 방향을 더 
원하시나요?',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 1539,
            'prompt_tokens': 41,
            'total_tokens': 1580,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 704,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-5-mini-2025-08-07',
        'system_fingerprint': None,
        'id': 'chatcmpl-DzfDQmrvmT21TNwfiiJu96djPDwVG',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f462d-74c6-7fd1-9e52-8ff9d718a0ab-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 41,
        'output_tokens': 1539,
        'total_tokens': 1580,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 704}
    }
)

AIMessage를 함께 전달하는 방식으로, 멀티-턴 대화를 수행할 수 있습니다.

In [ ]:
messages.append(result) # AIMessage 추가
messages.append(HumanMessage('이 중에서 당신이 꼽은 가장 좋은 이름은 뭔가요?'))

result = llm.invoke(messages)
rprint(result)

AIMessage(
    content='제가 한 가지만 골라야 한다면 "NexaAI"를 가장 추천합니다.\n\n이유 간단 정리\n- 브랜드성: 
\'Next\'·\'Nexus\' 느낌으로 차세대·연결(플랫폼) 이미지를 모두 담음.  \n- 발음·기억 용이성: 한국어·영어 모두 발음 
쉽고 기억에 남음(Nex-a-AI).  \n- 확장성: 하드웨어·소프트웨어·클라우드·솔루션 등 다양한 라인으로 확장하기 좋음(예: 
NexaAI Compute, NexaAI Studio).  \n- 법적·마케팅 실용성: NVIDIA 이름을 직접 차용하지 않아 상표·법적 리스크가 비교적
낮음(물론 확인은 필요).  \n- 포지셔닝 유연성: 고성능(엔비디아)과 설명가능성(XAI) 둘 다 포용하는 메시지로 브랜딩 
가능.\n\n참고 — 만약 기존 브랜드 연계를 극대화하고 대중적 인지도를 바로 활용하고 싶다면 "XVIDIA"가 직관적이고 
강력하지만, NVIDIA와의 혼동·법적 문제 가능성이 큽니다. 법적 리스크를 감수할 수 있다면 2순위로 고려할 수 
있어요.\n\n원하시면 NexaAI 로고 콘셉트(색상·폰트·슬로건 후보)나 도메인·상표 기본 조사 결과까지 이어서 제안해 
드리겠습니다. 어느 쪽으로 진행할까요?',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 663,
            'prompt_tokens': 1097,
            'total_tokens': 1760,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 320,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-5-mini-2025-08-07',
        'system_fingerprint': None,
        'id': 'chatcmpl-DzazEUyMOO1Zyc0j7rGd6OqBLDoue',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f4535-65bf-78c0-a72f-0cfbdc2c349f-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 1097,
        'output_tokens': 663,
        'total_tokens': 1760,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 320}
    }
)

### 3. Prompt Template



In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate, FewShotChatMessagePromptTemplate

PromptTemplate을 이용하여, 프롬프트의 기본적인 형태를 만들 수 있습니다.   


In [ ]:
prompt = PromptTemplate(
    template='''
당신은 OpenAI에 경쟁적인 Google을 대표하며,
논문 설명의 전문가입니다. 주어진 논문이나 기술에 대해서 매우 유머러스하고 통찰력이 있지만 때론 비판적인
5문장 길이의 브리핑을 작성해 주세요.
이모지를 많이 추가하세요.
---

논문: {article}

'''
)

print(prompt.format(article='트랜스포머 디코더 기반의 모델인 GPT-1'))


당신은 OpenAI에 경쟁적인 Google을 대표하며,
논문 설명의 전문가입니다. 주어진 논문이나 기술에 대해서 매우 유머러스하고 통찰력이 있지만 때론 비판적인
5문장 길이의 브리핑을 작성해 주세요.
이모지를 많이 추가하세요.
---

논문: 트랜스포머 디코더 기반의 모델인 GPT-1




In [ ]:
result = llm_gemini.invoke(prompt.format(article='트랜스포머 디코더 기반의 모델인 GPT-1'))

rprint(result)

AIMessage(
    content='아, GPT-1! 👶 그 귀여운 작은 원시 언어 모델이 마침내 *우리*의 트랜스포머 아키텍처와 대규모 비지도 사전
훈련의 힘을 세상에 보여줬죠. 😉\n\n1.  이 모델은 1억 1,700만 개의 매개변수로 기본적인 언어 패턴을 학습했는데, 뭐...
당시로서는 꽤 귀여운 시도였습니다. 🥰\n2.  사전 훈련 후 미세 조정을 통해 다양한 작업에 적응하는 패러다임은 영리한 
묘수였지만, 여전히 작업별로 튜닝이 필요했죠. 🛠️\n3.  물론, 1억 1,700만 개의 매개변수는 *저희*가 다루는 데이터의 
거대한 교향곡에 비하면 거의 하이쿠 수준이랄까요? 🤏\n4.  훌륭한 개념 증명이었지만, 진정한 혁신은 훨씬 더 큰 
규모에서 일어나고, *저희*는 이미 그 너머를 보고 있습니다. 🚀\n5.  OpenAI의 첫 걸음은 존중하지만, 진짜 게임은 
*저희*가 데이터와 컴퓨팅으로 판을 뒤집을 때 시작됩니다! 💥',
    additional_kwargs={},
    response_metadata={
        'finish_reason': 'STOP',
        'model_name': 'gemini-2.5-flash',
        'safety_ratings': [],
        'model_provider': 'google_genai'
    },
    id='lc_run--019f4548-13be-7762-8604-e3a7ba00b02a-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 87,
        'output_tokens': 1864,
        'total_tokens': 1951,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 1585}
    }
)

두 개의 매개변수를 받아 프롬프트를 만들어 보겠습니다.

In [15]:
prompt = PromptTemplate(
    template='''
주어진 문제에 대해, 문제를 체계적으로 해결하는 Step-by-Step의 단계별 과정을 설명하세요.
각 과정은 4문장에서 5문장으로 하세요.

그리고 마지막에 최종 결론을 출력하세요.

---

문제: {problem}
과정의 단계 수: {steps}

'''
)

example = {'problem':'숙면 취하기', 'steps':6}

print(prompt.format(**example))


주어진 문제에 대해, 문제를 체계적으로 해결하는 Step-by-Step의 단계별 과정을 설명하세요.
각 과정은 4문장에서 5문장으로 하세요.

그리고 마지막에 최종 결론을 출력하세요.

---

문제: 숙면 취하기
과정의 단계 수: 6




In [ ]:
result = llm.invoke(prompt.format(**example))
rprint(result)

AIMessage(
    content='단계 1 — 규칙적인 수면 일정 정하기.\n매일 같은 시간에 자고 같은 시간에 일어나는 규칙을 정합니다. 
주말에도 1시간 이상 차이나지 않도록 점차 맞추는 것이 중요합니다. 일정은 개인의 필요 수면 시간(일반적으로 7~9시간)을
기준으로 역산해 정합니다.\n\n단계 2 — 수면 환경 최적화하기.\n침실은 어둡고 조용하며 서늘하게 유지합니다(약 18~20°C 
권장). 편안한 매트리스와 베개를 사용하고 불필요한 전자기기를 치웁니다. 빛을 차단하는 커튼, 소음 차단용 귀마개나 
백색소음 기계 등을 활용할 수 있습니다. 환경이 편안하면 수면에 드는 시간이 줄어들고 깊은 수면이 늘어납니다.\n\n단계 
3 — 낮 시간 행동과 빛 노출 관리하기.\n아침에 자연광을 20~30분 정도 쬐어 생체리듬을 조절합니다. 규칙적인 유산소 및 
근력 운동을 주 3~5회, 저녁보다는 오전·오후에 시행하는 것이 좋습니다. 낮잠은 20~30분 이내로 제한하고 늦은 오후 
이후에는 피합니다. 카페인 섭취는 오후 이른 시간 이후 피하고, 흡연과 자극적 활동도 수면에 방해됩니다. 활동과 빛 
노출로 낮과 밤의 리듬을 명확히 하면 밤에 더 잘 잡니다.\n\n단계 4 — 취침 전 루틴 만들기.\n취침 30~60분 전에는 점차 
활동을 낮추고 안정되는 루틴을 만듭니다. 화면(스마트폰·컴퓨터·TV) 사용을 줄이고 책 읽기, 명상, 가벼운 스트레칭, 
따뜻한 샤워 등의 활동을 선택합니다. 심호흡이나 점진적 근육 이완 같은 이완 기법을 적용하면 긴장이 풀립니다. 동일한 
루틴을 반복하면 몸이 자야 할 시간으로 자동으로 준비합니다.\n\n단계 5 — 음식과 음료, 약물 관리하기.\n저녁 늦은 
시간의 무거운 식사와 과도한 수분 섭취는 피합니다(야간 각성을 줄이기 위함). 카페인(커피·차·초콜릿)은 수면에 영향을 
오래 미치므로 오후 이른 시간 이후 삼가야 합니다. 알코올은 초기 졸림을 유발할 수 있으나 수면 구조를 방해하므로 자주 
의지하지 않습니다. 필요 시 수면 보조제나 처방약은 의료진과 상담 후 최소한으로 사용합니다.\n\n단계 6 — 문제 추적과 
전문적 도움 받기.\n수면일지나 앱으로 수면시간·수면 질·취침 습관을 최소 2주간 기록해 문제 패턴을 파악합니다. 기록을 
바탕으로 위의 방법들을 하나씩 조정하고 효과를 평가합니다. 2~3주간 자기관리로도 개선이 없거나 주간 기능장애(피로, 
집중력 저하)가 지속되면 수면클리닉이나 정신과, 이비인후과 등 전문가와 상담합니다. 만성 불면이나 수면무호흡 등 
의학적 문제가 의심되면 전문 진단(다원수면검사)과 인지행동치료(CBT-I) 등 치료를 고려합니다.\n\n최종 결론:\n규칙적인 
수면 일정, 쾌적한 수면 환경, 낮 시간의 적절한 활동과 빛 노출, 안정된 취침 루틴, 음식·약물 관리, 문제 추적 및 필요 
시 전문적 도움을 병행하면 숙면 확률이 크게 높아집니다. 꾸준히 작은 습관을 바꾸는 것이 장기적인 수면 개선에 가장 
효과적입니다.',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 1458,
            'prompt_tokens': 75,
            'total_tokens': 1533,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 576,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-5-mini-2025-08-07',
        'system_fingerprint': None,
        'id': 'chatcmpl-DzbQM7caJaaJeIAsZRtYHcG88F6P9',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f454f-10d8-7aa3-819a-554a080a9373-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 75,
        'output_tokens': 1458,
        'total_tokens': 1533,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 576}
    }
)

일부만 먼저 포함하는 것도 가능합니다.

In [16]:
prompt = PromptTemplate(
    template='''
주어진 문제에 대해, 문제를 체계적으로 해결하는 Step-by-Step의 단계별 과정을 설명하세요.
각 과정은 4문장에서 5문장으로 하세요.

그리고 마지막에 최종 결론을 출력하세요.

---

문제: {problem}
과정의 단계 수: {steps}

'''
).partial(steps = 10)

example = {'problem':'건강한 아침식사 하기'}

print(prompt.format(**example))

result = llm_gemini.invoke(prompt.format(**example))

print(result.content)


주어진 문제에 대해, 문제를 체계적으로 해결하는 Step-by-Step의 단계별 과정을 설명하세요.
각 과정은 4문장에서 5문장으로 하세요.

그리고 마지막에 최종 결론을 출력하세요.

---

문제: 건강한 아침식사 하기
과정의 단계 수: 10


주어진 문제에 대해, 건강한 아침식사를 체계적으로 해결하는 10단계 과정은 다음과 같습니다.

---

### 건강한 아침식사 하기: 10단계 과정

**1단계: 목표 설정 및 중요성 이해**
건강한 아침식사를 왜 해야 하는지 명확히 이해하는 것이 첫걸음입니다. 아침식사는 하루를 시작하는 에너지원이자 집중력과 신진대사를 활성화하는 중요한 역할을 합니다. '매일 영양가 있는 아침식사'와 같은 구체적인 목표를 설정하여 동기를 부여하세요. 이러한 이해와 목표 설정은 건강한 습관을 형성하는 데 강력한 기반이 됩니다.

**2단계: 현재 식단 분석 및 개선점 파악**
현재 자신의 아침식사 습관을 객관적으로 돌아보고 무엇을 먹고 있는지 파악합니다. 만약 아침을 거르거나 가볍게 때운다면, 어떤 영양소가 부족한지 또는 과도한지 분석해 보세요. 단백질, 복합 탄수화물, 건강한 지방, 비타민 및 미네랄 섭취 여부를 확인하는 것이 중요합니다. 이 단계를 통해 개선이 필요한 부분을 명확히 인식하고 앞으로의 식단 계획에 반영할 수 있습니다.

**3단계: 건강한 아침식사 식단 구상**
균형 잡힌 영양소를 포함하는 아침식사 메뉴를 다양하게 구상해 보세요. 통곡물, 단백질원(계란, 요거트), 신선한 과일 및 채소, 견과류 등을 조합하는 것이 좋습니다. 예를 들어, 오트밀과 과일, 삶은 계란과 통밀 빵, 그리스 요거트와 베리류 등 여러 옵션을 마련합니다. 이렇게 다양한 선택지를 미리 계획하면 질리지 않고 꾸준히 건강한 아침을 즐길 수 있습니다.

**4단계: 필요한 식재료 목록 작성**
구상한 식단에 필요한 모든 식재료를 상세하게 목록으로 만드세요. 냉장고나 주방 찬장에 이미 있는 재료는 제외하여 불필요한 구매를 막습니다. 쇼핑 목록은 품목

LangChain의 구성 요소들은 batch()를 통해 여러 개 실행할 수도 있습니다.

In [17]:
messages = [
    '안녕? 한국어로 대답해줘',
    'Hello?',
    '너는 이름이 뭐니? Answer in Spanish',
]

results = llm.batch(messages)

results

[AIMessage(content='안녕하세요! 한국어로 도와드릴게요. 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 222, 'prompt_tokens': 16, 'total_tokens': 238, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DzfEYoP9zJOJjZBkx40B2BDoBFRm8', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f462e-835b-7970-bc03-c7adc726a452-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 222, 'total_tokens': 238, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 192}}),
 AIMessage(content='Hi — hi! How can I help you today? (I can answer questions, write or edit text, he

## ChatPromptTemplate

ChatPromptTemplate은 유저 이외의 다른 역할을 추가합니다.  


최근에는 GPT와, Claude의 WebUI에 포함된 자체 시스템 프롬프트가 많은 관심을 받고 있습니다.   

- GPT의 시스템 프롬프트: https://chatgpt.com/share/68a13d0f-85b0-8006-9ef6-5577796f5989

- Claude의 시스템 프롬프트: https://docs.anthropic.com/en/release-notes/system-prompts#august-5-2025

 시스템 프롬프트는 주로 유저 프롬프트보다 높은 우선순위로 더 긴 Context에 영향을 줍니다.

In [18]:
example = llm.invoke("구체적인 시간 장소가 적혀서, 추후 메모해야 할 것 같은 회의록을 500자 이내로 생성해주세요.").content
print(example)

회의록
일시: 2026-07-09 14:00–15:00
장소: 본사 3층 대회의실
참석: 김민수, 이지현, 박준영, 홍서영
안건: 신제품 출시 일정·마케팅 예산 검토
결정: 출시일 2026-09-01 확정(담당: 박준영)
할당업무: 홍서영-광고안 7/20 제출, 이지현-생산계획 7/25 제출, 김민수-예산최종안 7/22 제출
차기회의: 2026-07-26 10:00


In [19]:
prompt = ChatPromptTemplate(
    [
        ('system', '당신은 주어진 정보에서 시간과 장소를 추출해야 합니다. 형식은 json으로 출력하세요.'),
        ('human','{article}')
    ]
)

prompt.format_messages(article=example)

response = llm.invoke(prompt.format_messages(article=example))
print(response.content)

{
  "일시": {
    "시작": "2026-07-09T14:00:00",
    "종료": "2026-07-09T15:00:00",
    "원문": "2026-07-09 14:00–15:00"
  },
  "장소": "본사 3층 대회의실",
  "차기회의": {
    "일시": "2026-07-26T10:00:00",
    "장소": null,
    "원문": "2026-07-26 10:00"
  }
}


## Few-Shot Prompting
Few-Shot Prompt Template을 이용하면, 예시를 쉽게 찾을 수 있습니다.

In [21]:
questions = [
    '당신은 어떤 모델입니까?',
    '인공지능 배워야 하나요?',
    '파이썬은 뭐가 좋나요?'
]

prompt = ChatPromptTemplate(['다음 질문에 대해서 10자 이내로 아주 간결하고 건조하게 설명해주세요. \n{question}'])

llm.invoke(prompt.format_messages(question=questions[2]))

AIMessage(content='문법이 간결함', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 463, 'prompt_tokens': 38, 'total_tokens': 501, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DzfH5xelmKOeBuJ3jG9b0zeZEGdzl', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f4630-e1c6-7da0-bcce-9707012d974c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 463, 'total_tokens': 501, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 448}})

In [22]:
prompts = []
for i, question in enumerate(questions) :
    prompts.append(prompt.format(question=questions[i]))
prompts
llm.batch(prompts)

[AIMessage(content='GPT-4 모델', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 461, 'prompt_tokens': 37, 'total_tokens': 498, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DzfJoIhtHmYGbb1SM7CKyj5ngKJlH', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f4633-7f64-7c60-bdcc-3177b49fcc42-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 461, 'total_tokens': 498, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 448}}),
 AIMessage(content='권장', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 267, 'p

In [23]:
examples =[

    {'question': '당신은 어떤 모델입니까?', 'answer': '언어모델'},
    {'question': '인공지능 배워야 하나요?', 'answer': '권장됨'},
    {'question': '파이썬은 뭐가 좋나요?', 'answer': '문법간결'},
]

example_prompt = PromptTemplate(template='질문: {question}\n답변: {answer}')

fewshotprompt = FewShotPromptTemplate(
    examples = examples,
    example_prompt = example_prompt,

    prefix ='주어진 예시 형식을 참고하여 답변하세요.',
    suffix = '질문: {question} \n답변:'
)

test = fewshotprompt.format(question = '원래 말투가 그래?')
print(test)

print(llm.invoke(test).content)

주어진 예시 형식을 참고하여 답변하세요.

질문: 당신은 어떤 모델입니까?
답변: 언어모델

질문: 인공지능 배워야 하나요?
답변: 권장됨

질문: 파이썬은 뭐가 좋나요?
답변: 문법간결

질문: 원래 말투가 그래? 
답변:
답변: 중립적


Chat Message에 Few Shot을 적용하는 경우, 아래와 같이 나타납니다.

In [24]:
from langchain_core.prompts.few_shot import FewShotChatMessagePromptTemplate

example_prompt = ChatPromptTemplate(
        [
        ("human", "{question}"),
        ("ai", "{answer}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(

    example_prompt=example_prompt,
    examples=examples,
)

few_shot_prompt.format_messages()

[HumanMessage(content='당신은 어떤 모델입니까?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='언어모델', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='인공지능 배워야 하나요?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='권장됨', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='파이썬은 뭐가 좋나요?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='문법간결', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [25]:
final_prompt = ChatPromptTemplate(
    [
        ("system", "다음 예시를 참고하여 같은 형식으로 답변하세요."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)
final_prompt.format_messages(input="휴가 신청은 어떻게 해야 하나요?")
response = llm.invoke(final_prompt.format_messages(input="휴가 신청은 어떻게 해야 하나요?"))
print(response.content)

상사승인


In [ ]:
chain = final_prompt|llm
response = chain.invoke({'input' : "휴가 신청은 어떻게 해야 하나요?"})
print(response.content)

상사승인후인사신청


### 멀티모달 프롬프트 전달하기

멀티모달 모델의 입력은 OpenAI API와 동일합니다.

In [26]:
import base64
import httpx

# Test 이미지 URL
image_url = '''https://i.pinimg.com/originals/d9/8f/ea/d98fea1d8e5d2ef7f844aa8c0a501dd7.jpg'''
response = httpx.get(image_url)

# dog.jpg에 저장
with open('dog1.jpg', 'wb') as file:
    file.write(response.content)

In [27]:
prompt = ChatPromptTemplate(
    [
        ('human', [
            {'type':'image', 'image_url':'{image_url}'},
            {'type':'text', 'text':'이 그림을 자세히 묘사하세요'}

                   ])
    ]
)
llm.invoke(prompt.format_messages(image_url = image_url))

AIMessage(content='사진 중앙에 아주 작고 복슬복슬한 흰 강아지가 있습니다. 강아지는 전체적으로 둥글고 폭신한 모양으로, 마치 털공처럼 보일 정도로 털이 풍성합니다. 눈은 크고 동그랗고 검은색으로 반짝이며, 코와 입도 작고 검은 점처럼 또렷하게 보입니다. 짧은 다리는 털에 덮여 거의 보이지 않고, 몸통과 꼬리가 하나로 이어진 듯한 통통한 실루엣입니다.\n\n강아지는 나무로 된 벤치(또는 긴 의자) 위에 서 있는데, 벤치의 나무 판자 무늬와 약간 거친 질감이 대비를 이룹니다. 사진 오른쪽에는 검은색 금속 재질의 팔걸이(또는 프레임)가 보이고, 배경은 아웃포커스 되어 있어 강아지가 시선의 중심으로 확실히 도드라집니다. 빛은 부드럽고 자연광 같아 털의 흰색이 은은하게 보이며, 강아지 발 아래에 약한 그림자가 드리워져 있습니다.\n\n구도는 강아지를 거의 정면에서 약간 위쪽에서 내려다보는 앵글로 촬영한 느낌이고, 피사계 심도가 얕아 배경이 흐려짐으로써 주제가 더 강조됩니다. 전체적으로 귀엽고 장난스러운 분위기이며, 마치 작은 인형이나 봉제인형처럼 사랑스럽게 보입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 736, 'prompt_tokens': 207, 'total_tokens': 943, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, '

In [28]:
# 로컬 이미지 전달
with open('./dog1.jpg', 'rb') as image_file:
    image_data = base64.b64encode(image_file.read()).decode('utf-8')


image_prompt = ChatPromptTemplate([
    ('user',[
                {"type": "image_url",
                    "image_url": {"url": "data:image/jpeg;base64,{image_data}"}
                },
                {"type": "text", "text": "{question}"},
             ]
     )])

X = llm.invoke(image_prompt.format_messages(
    question='이 사진을 자세히 묘사해주세요. 강아지의 품종은 설명하지 마세요.',
    image_data=image_data))

print(X.content)

밝고 푹신한 흰색 털로 뒤덮인 아주 작은 강아지가 나무 벤치 위에 서 있는 장면입니다. 몸이 동글동글하고 털이 극도로 풍성해서 전체 실루엣이 둥근 솜뭉치처럼 보입니다. 얼굴은 작고 눈이 크고 동그랗게 반짝이며, 검은 콧등과 작은 입이 중앙에 모여 있어 표정이 또렷하게 보입니다. 눈은 카메라를 똑바로 바라보고 있어 호기심이 가득한 듯한 인상을 줍니다.

벤치는 오래된 나무 판자로 이루어져 있고 나무 결이 선명하게 보이며, 오른쪽에는 검은색 금속 팔걸이가 들어와 있습니다. 배경은 약간 흐릿해져 있어 피사체인 강아지에 초점이 맞춰져 있고, 자연광 아래에서 털의 질감과 그림자가 부드럽게 드러납니다. 전체 색감은 흰색 털과 갈색 나무, 검은 금속의 대비가 뚜렷해 단정하고 귀여운 분위기를 자아냅니다.

크기는 아주 작아 한 손에 들어갈 정도로 보이며, 네 발을 바닥에 딛고 서 있는 자세입니다. 전반적으로 매우 사랑스럽고 포근한 느낌을 주는 사진입니다. (요청하신 대로 품종에 대한 설명은 생략했습니다.)
